## The conceptual framework: what is the bottleneck?

The right question for any task is: **what is the scarce resource?**

- **I/O-bound**: the scarce resource is the _external system_ — network, disk, database. The CPU is mostly idle, waiting.
- **CPU-bound**: the scarce resource is _compute_ — the CPU is running flat out. There is no waiting.

Everything follows from this.

---

## Threading for I/O-bound — why it works well

The CPU is idle most of the time anyway. Threading exploits that idle time:

- Thread A blocks on a network call → CPU would be doing nothing → Thread B runs instead.
- The GIL is released during the syscall, so Thread B gets full Python execution.
- You can create **hundreds or thousands of threads** cheaply. A thread is just a stack (~1 MB) and a register snapshot. Most threads are in WAITING state and cost almost no CPU — they're just queued in the kernel waiting for I/O to complete.
- The bottleneck is the I/O subsystem (network bandwidth, disk IOPS), not the CPU. More threads = more concurrent I/O in flight, and the hardware can service them.

---

## Threading for CPU-bound — why it's bad

The CPU is the scarce resource. You want more CPU power. Threading gives you none:

- The GIL means only one thread executes Python bytecode at a time — always.
- With 4 threads on a 4-core machine, all 4 cores are available, but Python can only use 1. You've paid for 3 threads' worth of overhead (stack memory, OS context switches, GIL contention) and gotten zero parallelism.
- It can be **slower** than single-threaded: threads fight over the GIL, the OS wastes time context-switching between them, and no extra computation happens.

---

## Multiprocessing for CPU-bound — why it works

Each process has its own GIL (the GIL is per-interpreter). 4 processes on 4 cores = 4 Python interpreters running bytecode truly in parallel. The CPU saturation you needed.

---

## Multiprocessing for I/O-bound — why it's bad

You're right about the core count limit — but there's more to it:

- **Process creation is expensive**: spawning a process costs ~100ms and megabytes of memory (the interpreter, loaded modules, a full copy of the heap). Thread creation is microseconds and kilobytes.
- **You're limited to tens of processes, not thousands**. For I/O-bound work you might want 500 concurrent HTTP requests. 500 threads: fine. 500 processes: impractical.
- **The CPU isn't the bottleneck** — adding more processes doesn't help if the wait is on the network. A single process with many threads can keep just as many I/O requests in flight.
- **IPC is expensive**: sharing data between processes requires serialization (pickling) and explicit channels (pipes, queues). Threads share heap directly.

Summary: multiprocessing adds CPU resources. I/O-bound tasks don't need CPU resources — they need _waiting capacity_. Threads provide that cheaply. Processes solve the wrong problem expensively.

---

## Adding asyncio — completing the picture

asyncio targets the same problem as threading (I/O-bound concurrency) but from a completely different angle: instead of running multiple threads and letting the OS switch between them, you run **one thread** and **write code that explicitly hands off control** at I/O boundaries.

### Where asyncio beats threading for I/O-bound work

||`threading`|`asyncio`|
|---|---|---|
|**Race conditions**|Real risk — OS can switch threads between any two bytecodes|None for normal shared state — a coroutine runs uninterrupted until it hits `await`. You know exactly where hand-off happens.|
|**Concurrency scale**|Hundreds of threads before OS overhead and memory hurt|Tens of thousands of coroutines — a coroutine is just a Python object (~1-2 KB), no OS thread involved|
|**Context switch cost**|Kernel-level (save/restore registers, mode switch)|User-level (resume a Python generator) — much cheaper|
|**Synchronization overhead**|Locks, semaphores needed for shared state|Usually not needed — only one coroutine runs at a time|

#### Race Condition in `asyncio`
Your instinct about race conditions is the key one. With threading, the OS can preempt a thread between _any two bytecodes_ — even something as simple as `x = x + 1` is not safe. With asyncio, a coroutine runs until it explicitly `await`s, so you know shared state won't change underneath you mid-operation (as long as you don't `await` in the middle of a critical section). [link](asyncio\race_cond_trap.ipynb)

### Where threading beats asyncio

||`threading`|`asyncio`|
|---|---|---|
|**Legacy/sync libraries**|Any sync library works — `requests`, old DB drivers, anything|If a library is sync and blocking, it **blocks the entire event loop** — all other coroutines freeze|
|**One bad call ruins everything**|A blocking call in one thread only freezes that thread|A `time.sleep()` or `requests.get()` in a coroutine blocks everything|
|**Mental model**|Familiar, maps to OS concepts|Requires internalizing async/await, event loop, cooperative yielding|

### The summary

```shell
I/O-bound, many concurrent tasks  →  asyncio  (scale, no race conditions)
I/O-bound, legacy sync libs       →  threading (can use any sync library)
CPU-bound, need parallelism       →  multiprocessing (bypass GIL, true parallelism)
CPU-bound + I/O-bound mixed       →  asyncio + ProcessPoolExecutor
```

> `asyncio` didn't replace threading — it replaced the need to manage concurrency _through the OS scheduler_. You trade "the OS decides when to switch" for "I decide when to switch (at every `await`)". That trade buys you scale and safety, but costs you the freedom to use blocking code anywhere.

---

## Mental model

### Process

A **process** is a program in execution. The OS creates a process when you run a program, giving it:

1. **Isolated virtual memory** — divided into four regions:

|Region|What lives there|
|---|---|
|**Code (Text) segment**|The compiled instructions (read-only)|
|**Data segment**|Global and static variables|
|**Heap**|Dynamically allocated memory (`malloc`, `new`, Python objects). Grows upward. Managed explicitly (or by GC).|
|**Stack**|One per thread. Call frames: local variables, function parameters, return addresses. Grows downward. Managed automatically (LIFO).|

2. **System resources** — the OS assigns these **exclusively** to the process:

- **CPU time slots** (scheduled by OS)
- Open file descriptors (files, sockets, pipes)
- Memory pages
- Signal handlers, environment variables, current working directory
- PID (Process ID) — unique OS-wide identifier

"Exclusive" means: no other process can directly touch these resources. The OS reclaims all of them on termination. The one exception worth knowing: processes _can_ deliberately share memory via `mmap` / shared memory segments — but this is opt-in IPC, not the default.

**Hardware interaction:** when a program needs hardware (disk, network, GPU), it calls into the OS via a **system call** (syscall). The CPU switches from user mode → kernel mode, the OS services the request, then switches back.

---

### Registers

**CPU registers** are tiny, ultra-fast storage cells inside the CPU itself — not RAM. They hold the values the CPU is actively operating on _right now_: operands for arithmetic, memory addresses being accessed, etc.

Each thread has a **saved register snapshot** (sometimes called a "register context" or "CPU context"). When the OS preempts a thread, it dumps all current register values into that snapshot. When the thread resumes, those values are loaded back into the CPU registers — the thread continues as if nothing happened.

The two most important saved registers:

|Register|Role|
|---|---|
|**PC (Program Counter)**|Address of the _next instruction_ to execute|
|**SP (Stack Pointer)**|Address of the top of the current stack frame|

The PC is what makes "resume where you left off" work.

---

### Thread

- A thread is the **unit of CPU scheduling** within a process.
- A process starts with one thread (main thread) and can create more.
- All threads in a process share: heap, code, data segment, open file descriptors, PID.
- Each thread owns: TID, program counter, register snapshot, and its own **stack**.

> **The stack isolation is critical**: each function call pushes a _stack frame_ (local variables + return address). Threads calling the same function simultaneously each have their own frame on their own stack — they don't clobber each other's locals.

---

### 1) How many processes and threads?

|Program type|Processes|Threads|Notes|
|---|---|---|---|
|**Sync**|1|1 (main)|One thing at a time|
|**`threading`**|1|1 + N workers|N threads share heap — GIL limits true CPU parallelism|
|**`asyncio`**|1|1|N coroutines on one thread — event loop switches between them|
|**`multiprocessing`**|1 + N|1+ per process|True CPU parallelism — separate memory per process|
|**`asyncio` + `run_in_executor`**|1|1 event loop + thread/process pool workers|Mixed: async I/O + offloaded blocking work|

The crucial mental model: **asyncio is not about threads at all** — it's N coroutines time-sharing one thread via the event loop.

---

### 2) Threading, I/O-bound ops, preemption — exactly what happens?

You're right that threading uses **preemptive multitasking**. But there are actually _two_ ways a thread stops running, and I/O triggers the second one:

**Mechanism A — timer preemption (what "preemptive" usually means):**  
The OS gives each thread a time quantum (~1-10ms). When the timer fires, a hardware interrupt interrupts the CPU, the kernel saves the thread's register snapshot, and picks the next READY thread. The thread had no say in this.

**Mechanism B — voluntary blocking on a syscall (what I/O actually does):**  
When a thread calls `socket.recv()` or `file.read()`:

1. Thread executes the syscall → enters kernel mode
2. Kernel sees the data isn't ready yet → marks thread as **WAITING** (not runnable)
3. Kernel returns control to the scheduler immediately
4. Scheduler picks another READY thread and runs it
5. Later: the NIC/disk fires a **hardware interrupt** when data arrives
6. The interrupt handler runs in the kernel, finds the waiting thread, marks it **READY**
7. Scheduler eventually picks it up, restores register snapshot → thread resumes at the instruction after the syscall

**Who notifies who:** Hardware → kernel interrupt handler → scheduler. The thread itself is passive — it just wakes up where it blocked.

**Python-specific (GIL):** Python normally holds the GIL per thread. But the GIL is **released before a blocking syscall** and **re-acquired after**. So during step 4 above, another Python thread can acquire the GIL and run Python bytecode. This is why threading works well for I/O-bound Python code despite the GIL.

### Preemptive (threading) vs cooperative (asyncio) — side by side

||`threading` (preemptive)|`asyncio` (cooperative)|
|---|---|---|
|**Who decides to switch?**|OS (timer interrupt or blocking syscall)|The coroutine itself (`await`)|
|**Can a coroutine/thread block others?**|A CPU-spinning thread still gets preempted by the OS timer|A coroutine that never `await`s **blocks the entire event loop**|
|**I/O mechanism**|Blocking syscall → thread suspended by kernel|Non-blocking syscall + `epoll`/`select` → event loop notified when ready|
|**Switch cost**|Full kernel context switch (save/restore registers, kernel→user mode)|Cheap: just resumes a Python generator|
|**Memory**|Each thread needs its own stack (~1MB)|Coroutines are just Python objects (~few KB)|

Your asyncio description is exactly right: one process, one thread, one event loop. When a coroutine hits `await`, it gives up control deliberately — the event loop runs the next ready coroutine. The OS is never involved in the switching. The event loop uses `epoll`/`select` to ask the OS "tell me when _any_ of these sockets have data" — a single non-blocking call — and when the OS replies, the event loop resumes the right coroutine.

The GIL is released during blocking I/O operations: network calls, disk reads, `time.sleep()`, database queries. While one thread waits on I/O, **others can run**.

GIL is released for another thread, and when the result is ready for the first thread it is marked as `READY` the scheduler picks it up, restores its register snapshot and resumes it, but the thread still can't run Python bytecode yet — it first has to re-acquire the GIL. If another thread currently holds it, the resuming thread blocks again (this time waiting for the GIL, not for I/O).

It's still **only one thread running bytecode at any moment**. When Thread A releases the GIL for I/O, Thread B acquires the GIL and runs. Thread A is sitting in a syscall, not executing bytecode. The win is that Thread A's dead waiting time is filled by Thread B doing real work — not that both run simultaneously.

```shell
I/O-bound work  → threading helps SIGNIFICANTLY (GIL released during I/O)
CPU-bound work  → threading makes it WORSE (GIL contention overhead)
```
This is also why threading still has contention overhead even for pure I/O-bound work — multiple threads finishing I/O at the same time all race to re-acquire the GIL.

`time.sleep()` follows the same path except the wake-up trigger is a **kernel timer** (not a hardware interrupt from a device). The GIL is still released during the sleep.
```shell
thread calls socket.recv()
  → CPython releases GIL          ← other threads can now run Python
  → thread enters blocking syscall (kernel WAITING state)
  → OS scheduler runs another thread

... NIC receives data ...
  → hardware interrupt fires
  → kernel interrupt handler: marks thread READY

OS scheduler picks up the thread
  → restores register snapshot (PC, SP, etc.)
  → thread exits kernel mode, back in user mode
  → thread tries to re-acquire GIL
      → if GIL is free: acquires it, runs Python
      → if GIL is held: waits here until the holder releases it
```

---